In [ ]:
# Packages
import pandas as pd
import numpy as np
import json
import requests
import os
from functools import reduce
from tqdm import tqdm
import functools as ft
import math
pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')

print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
key = dict_api[user]
key = '?registrationkey={}'.format(key)

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Import indicators table
df_indicators = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Indicators')

# Set parameters for querying BLS data
indicator_name = df_params[df_params['Type'] == 'indicator_name']['Input'].values[0]
survey         = df_params[df_params['Type'] == 'survey'        ]['Input'].values[0]
geography      = df_params[df_params['Type'] == 'geography'     ]['Input'].values[0]
percentages    = df_params[df_params['Type'] == 'percentages'   ]['Input'].values[0]
year_start     = df_params[df_params['Type'] == 'year_start'    ]['Input'].values[0]
year_end       = df_params[df_params['Type'] == 'year_end'      ]['Input'].values[0]

# View
print(indicator_name)
print(survey)
print(geography)
print("Percentages: " + percentages)
print(year_start)
print(year_end)

print('')
print('Export parameters:' )
print('')

df_indicators = df_indicators[df_indicators['Indicator'] == indicator_name]

# Set parameters for export file
project     = df_indicators['Project'            ].values[0]
export_loc  = df_indicators['Export Location'    ].values[0]
folder      = df_indicators['Folder'             ].values[0]

# View
print('Project:             ' + project              )
print('Export Location:     ' + export_loc           )
print('Folder name:         ' + folder               )


In [ ]:
# Import data by geography
path_in = os.path.join(path_main, export_loc, indicator_name + ' ' + folder)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties BLS ' + survey + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO BLS ' + survey + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

try: 
    df_msa = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'MSA')
    display(df_msa.head(3))
except Exception as e: print(e)


if indicator_name == 'Jobs_3':
    try: 
        df_msa1 = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'Government and Private')
        df_msa2 = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'Goods and Services')
        df_msa1 = df_msa1[df_msa1['Sector'] != 'All']
        df_msa2 = df_msa2[df_msa2['Sector'] != 'All']
        df_msa = pd.concat([df_msa1, df_msa2])
        display(df_msa.head(3))
    except Exception as e: print(e)
    


In [ ]:
if geography == 'MSA':      
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]
    df_msa['date_'] = df_msa['date_'].astype('str')
    display(df_msa.head())

In [ ]:
if geography == 'MSA':
    name_output_MSA_csv = [indicator_name, '_MSA_', 'BLS_', survey,'.csv']
    name_output_MSA_csv = "".join(name_output_MSA_csv)


In [ ]:
# Set file path for exporting
path_out_csv  = os.path.join(path_agol, indicator_name)
print('CSV files exported here: ' + path_out_csv)

if geography == 'Counties':
    df_counties.to_csv(os.path.join(path_out_csv, name_output_counties_csv), index = False)
    df_mpo     .to_csv(os.path.join(path_out_csv, name_output_MPO_csv     ), index = False)

if geography == 'MSA':
    df_msa.to_csv(os.path.join(path_out_csv, name_output_MSA_csv), index = False)